In [1]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("processed_dataset.csv")

In [3]:
def train_f1_models(df):

    # ============================================================
    # FEATURES & TARGET
    # ============================================================

    features = [
        "POINTS",
        "LAPS",
        "MILLISECONDS",
        "WEATHER_cloudy",
        "OVERTAKEN_POSITIONS_TOTAL",
        "DNF_COUNT",
        "LAPMEAN",
        "PS_COUNT",
        "SC_COUNT",
        "DRIVER_ENCODED",
        "RACE_ENCODED"
    ]

    target = "SCORE"

    # ============================================================
    # CLEAN DATA
    # ============================================================

    model_df = df[features + [target]].copy()
    model_df = model_df.replace([np.inf, -np.inf], np.nan)
    model_df = model_df.dropna()

    X = model_df[features]
    y = model_df[target].values

    # ============================================================
    # SPLIT
    # ============================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # ============================================================
    # SCALING
    # ============================================================

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ============================================================
    # TENSORS
    # ============================================================

    X_train_tensor = torch.tensor(
        X_train_scaled,
        dtype=torch.float32
    )

    X_test_tensor = torch.tensor(
        X_test_scaled,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32
    ).view(-1, 1)

    y_test_tensor = torch.tensor(
        y_test,
        dtype=torch.float32
    ).view(-1, 1)

    # ============================================================
    # DEVICE
    # ============================================================

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # ============================================================
    # DATALOADER
    # ============================================================

    train_dataset = TensorDataset(
        X_train_tensor.to(device),
        y_train_tensor.to(device)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True
    )

    # ============================================================
    # MODEL
    # ============================================================

    model = F1NeuralNetwork(
        input_size=X_train.shape[1]
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-5
    )

    # ============================================================
    # TRAIN
    # ============================================================

    epochs = 300
    losses = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for batch_X, batch_y in train_loader:

            optimizer.zero_grad()

            predictions = model(batch_X)

            loss = criterion(
                predictions,
                batch_y
            )

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        losses.append(
            running_loss / len(train_loader)
        )

    # ============================================================
    # NN PREDICTIONS
    # ============================================================

    model.eval()

    with torch.no_grad():

        y_pred_nn = model(
            X_test_tensor.to(device)
        )

    y_pred_nn = (
        y_pred_nn.cpu()
        .numpy()
        .flatten()
    )

    nn_metrics = {
        "MAE": mean_absolute_error(y_test, y_pred_nn),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_nn)),
        "R2": r2_score(y_test, y_pred_nn)
    }

    # ============================================================
    # RANDOM FOREST
    # ============================================================

    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    rf_pred = rf_model.predict(X_test)

    rf_metrics = {
        "MAE": mean_absolute_error(y_test, rf_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, rf_pred)),
        "R2": r2_score(y_test, rf_pred)
    }

    # ============================================================
    # EXTRA TREES
    # ============================================================

    et_model = ExtraTreesRegressor(
        n_estimators=400,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )

    et_model.fit(X_train, y_train)

    et_pred = et_model.predict(X_test)

    et_metrics = {
        "MAE": mean_absolute_error(y_test, et_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, et_pred)),
        "R2": r2_score(y_test, et_pred)
    }

    # ============================================================
    # GRADIENT BOOSTING
    # ============================================================

    gb_model = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=5,
        random_state=42
    )

    gb_model.fit(X_train, y_train)

    gb_pred = gb_model.predict(X_test)

    gb_metrics = {
        "MAE": mean_absolute_error(y_test, gb_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, gb_pred)),
        "R2": r2_score(y_test, gb_pred)
    }

    # ============================================================
    # COMPARISON
    # ============================================================

    comparison = pd.DataFrame({
        "Model": [
            "PyTorch Neural Network",
            "Random Forest",
            "Extra Trees",
            "Gradient Boosting"
        ],
        "MAE": [
            nn_metrics["MAE"],
            rf_metrics["MAE"],
            et_metrics["MAE"],
            gb_metrics["MAE"]
        ],
        "RMSE": [
            nn_metrics["RMSE"],
            rf_metrics["RMSE"],
            et_metrics["RMSE"],
            gb_metrics["RMSE"]
        ],
        "R2": [
            nn_metrics["R2"],
            rf_metrics["R2"],
            et_metrics["R2"],
            gb_metrics["R2"]
        ]
    }).sort_values(
        by="R2",
        ascending=False
    )

    # ============================================================
    # FEATURE IMPORTANCE
    # ============================================================

    importance_df = pd.DataFrame({
        "Feature": features,
        "Importance": et_model.feature_importances_
    }).sort_values(
        by="Importance",
        ascending=False
    )

    return {
        "model": model,
        "scaler": scaler,
        "comparison": comparison,
        "feature_importance": importance_df,
        "losses": losses,
        "rf_model": rf_model,
        "et_model": et_model,
        "gb_model": gb_model
    }